# Denoising Autoencoder（DAE）によるノイズ除去

---
## 目的
Denoising Autoencoder (DAE) を用いて，画像に付与したノイズを除去する仕組みを理解する．`autoencoder.ipynb`のAEとの違い（教師信号の与え方）に注目する．MNISTデータセットに人為的にスパイクノイズを付与し，DAEによるノイズ除去を行う．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import numpy as np
from time import time
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Denoising Autoencoderとは
`autoencoder.ipynb`のAEは，入力画像`x`をそのまま教師信号として，`model(x) ≈ x`となるように学習しました．DAEは，入力にはノイズを付与した画像$\tilde{x}$を，教師信号には元のノイズのない画像$x$を用いることで，`model(\tilde{x}) ≈ x`となるように学習します．

これにより，ネットワークは単に入力を丸暗記するのではなく，「画像を構成する本質的な特徴（数字の形など）」を残しつつノイズを取り除くように学習されるため，ノイズ除去に応用できるだけでなく，より頑健な特徴表現を獲得できると考えられています．

## データセットの作成
`torchvision.datasets.MNIST`を継承した`DenoiseMNIST`クラスを定義します．`__getitem__`では，元の画像`denoised`（教師信号）にランダムなマスクを掛けてスパイクノイズを付与した画像`noise`（ネットワークへの入力）を作成し，`(noise, denoised)`のペアを返します．

In [ ]:
class DenoiseMNIST(torchvision.datasets.MNIST):
    def __init__(self, root, train=True, transform=None, download=False):
        super().__init__(root, train=train, transform=transform, download=download)
        self.noise_ratio = 0.25  # ノイズを付与する画素の割合

    def __getitem__(self, index):
        denoised = self.data[index]  # ノイズを含まない元の画像（教師信号）

        # ランダムなマスクを掛けてスパイクノイズを付与する
        mask = np.random.binomial(size=denoised.size(), n=1, p=1.0 - self.noise_ratio)
        mask = torch.from_numpy(mask.astype(np.uint8))
        noise = denoised * mask

        if self.transform is not None:
            noise = self.transform(Image.fromarray(noise.numpy()))
            denoised = self.transform(Image.fromarray(denoised.numpy()))

        return noise, denoised


train_data = DenoiseMNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_data = DenoiseMNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)

# ノイズ付与前後の画像を表示して確認する
cols = 10
fig, axes = plt.subplots(2, cols, figsize=(14, 2.8))
for c in range(cols):
    axes[0, c].imshow(train_data[c][0].reshape(28, 28), cmap='gray'); axes[0, c].axis('off')
    axes[1, c].imshow(train_data[c][1].reshape(28, 28), cmap='gray'); axes[1, c].axis('off')
axes[0, 0].set_ylabel('noise'); axes[1, 0].set_ylabel('denoised')
plt.tight_layout()
plt.show()

## ネットワークモデルの定義
ネットワーク構造は`autoencoder.ipynb`のAEと全く同じ（Encoder・Decoderともに全結合層3層，`784 ⇄ 256 ⇄ 100 ⇄ 潜在変数の次元数`）です．構造の詳細は`autoencoder.ipynb`を参照してください．DAEは，ネットワーク構造ではなく**学習時に与える入力・教師信号の組み合わせ方**が異なる手法です．

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, n_hidden):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100), nn.ReLU(inplace=True),
            nn.Linear(100, n_hidden),
        )
        self.decoder = nn.Sequential(
            nn.Linear(n_hidden, 100), nn.ReLU(inplace=True),
            nn.Linear(100, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 28 * 28), nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.encoder(x)
        return self.decoder(h)

## ネットワークの作成
`autoencoder.ipynb`と同じ設定でネットワークを作成します．潜在変数の次元数は，後ほど潜在空間を可視化するために`2`とします．

In [ ]:
hidden_num = 2

model = AutoEncoder(n_hidden=hidden_num).to(device)
optimizer = torch.optim.Adam(model.parameters())

## 学習
ミニバッチサイズを100，学習エポック数を20として学習します．誤差関数には`autoencoder.ipynb`と同じ`MSELoss`を使用しますが，ネットワークにはノイズ画像`image`を入力し，ノイズのない画像`label`との誤差を計算する点が異なります．

In [ ]:
batch_size = 100
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
criterion = nn.MSELoss().to(device)

model.train()
start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for image, label in train_loader:
        image = image.view(image.size(0), -1).to(device)  # ノイズ画像（入力）
        label = label.view(label.size(0), -1).to(device)  # ノイズのない画像（教師信号）

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print(f'epoch: {epoch}, mean loss: {sum_loss / len(train_loader):.4f}, elapsed_time: {time() - start:.4f}')

## テスト
学習したネットワークを用いてデノイジングを行い，結果を可視化します．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False)

model.eval()
test_result, test_gt = [], []
with torch.no_grad():
    for count, (image, label) in enumerate(test_loader):
        image = image.view(image.size(0), -1).to(device)

        y = model(image)

        test_result.append(y.cpu())
        test_gt.append(label)

        if count == 9:
            break

cols = 10
fig, axes = plt.subplots(2, cols, figsize=(14, 2.8))
for c in range(cols):
    axes[0, c].imshow(test_result[c].reshape(28, 28), cmap='gray'); axes[0, c].axis('off')
    axes[1, c].imshow(test_gt[c][0].reshape(28, 28), cmap='gray'); axes[1, c].axis('off')
fig.suptitle('denoise result (top) / ground truth (bottom)')
plt.show()

## 潜在空間の可視化
`autoencoder.ipynb`と同様に，潜在空間の2次元ベクトルを格子状に作成し，Decoderへ入力して画像を生成することで，DAEが獲得した潜在空間を可視化します．

※ この処理は`hidden_num = 2`の場合のみ動作します．

In [ ]:
nv = 25
value1 = np.linspace(-2.0, 2.0, nv)
value2 = np.linspace(-2.0, 2.0, nv)

plot_array = np.zeros([28 * nv, 28 * nv], dtype=np.float32)
model.eval()
with torch.no_grad():
    for i, yi in enumerate(value1):
        for j, xj in enumerate(value2):
            xx = torch.tensor([[yi, xj]], dtype=torch.float32).to(device)
            output = model.decoder(xx)
            output = output.view(28, 28).cpu().numpy()
            plot_array[(nv - i - 1) * 28:(nv - i) * 28, j * 28:(j + 1) * 28] = output

plt.figure(figsize=(10, 10))
plt.imshow(plot_array, origin='upper', cmap='gray')
plt.tight_layout()
plt.show()

## 課題

1. スパイクノイズの割合（`noise_ratio`）を変更して学習し，ノイズの強さに対する復元性能の変化を確認してください．
2. `autoencoder.ipynb`で学習した（ノイズを一切見せていない）AEに，本ノートブックと同じノイズ画像を入力して比較してください．DAEの方がノイズに頑健な理由を考察してください．
3. スパイクノイズではなく，ガウスノイズ（`torch.randn`を用いて画素値に正規分布に従うノイズを加算する）を付与するデータセットに変更し，同様に学習・評価してください．